# Which detector should I build?

**Author:** Antoine ([@antowan](https://github.com/antowan))

Accuracy and inference time were measured on an EX5 (Jetson Orin NX 16 GB). Section 3 compares its live camera counts with an EX3 (Jetson Orin Nano 8 GB). You can read the results without running code or connecting a station.

**Accuracy and inference time** (sections 1 and 2):

| Model | mAP 50:95, 80 classes (published) | mAP 50:95, car / person / truck | FP16 ms per image, batch 1 | FP16 ms per image, batch 8 |
|---|---:|---:|---:|---:|
| YOLO26n | 39.92 (40.1) | 40.67 | 5.3 | 3.4 |
| YOLOv10n | 38.34 (38.5) | 40.54 | 5.5 | 3.8 |
| YOLOv7-tiny | 37.13 (n/a) | 39.41 | 8.4 | 4.7 |

**Live cameras per pipeline, no video output** (section 3): each 30 FPS H.264 RTSP camera had to sustain at least 29 processed FPS. The EX3 was not tested with 1080p inputs.

| Model | EX3 720p | EX5 720p | EX5 1080p |
|---|---:|---:|---:|
| YOLO26n | 5† | 7 | 7 |
| YOLOv10n | 5 | 6† | 6 |
| YOLOv7-tiny | 3 | 4 | 4 |

† A higher count passed in some attempts and failed in others; the table keeps the lower count.

These measurements show different trade-offs, not a universal model ranking. YOLO26n and YOLOv10n differ by just **0.13 mAP points** on car, person and truck, while their speed and camera results are specific to the tested hardware and workload. Compare the classes and operating conditions that matter to your application, review each model’s license, and validate your choice with your own video.

## 1. How well does it detect?

A detector must name the object **and** put its box in the right place. Intersection over union (IoU) measures how much the predicted and labelled boxes overlap. **mAP 50:95** averages precision across classes and ten IoU thresholds, from 0.50 to 0.95. A score of 40 is not “40% of objects detected.”

Use **80 classes** to compare with published COCO results. On all 5,000 COCO val2017 images at 640 px, the raw FP16 engines scored within 0.2 points of the [YOLO26n end-to-end reference (40.1) and YOLOv10n paper reference (38.5)](../doc/benchmarks/orin-nx-16gb-jp6.0/comparison.md#accuracy). We found no directly comparable official 640 px result for these YOLOv7-tiny weights. The **three-class** column scores the same predictions for car, person and truck only; it is not a published benchmark or a test of the deployed callback.

### Accuracy by class

Scores of the TensorRT FP16 engine on all 5,000 COCO val2017 images, in percent. **AP 50** accepts a box from 50% overlap, a looser test than mAP 50:95. Read across a row to see whether a similar overall score hides a difference on a class you care about. These evaluations use labelled images rather than tracked video.

| Model | Published COCO mAP 50:95 | Measured mAP 50:95 (80 classes) | AP 50 | mAP 50:95 (car, person, truck) | AP car | AP person | AP truck |
|---|---:|---:|---:|---:|---:|---:|---:|
| YOLO26n | 40.1 | 39.92 | 55.57 | 40.67 | 37.02 | 52.69 | 32.32 |
| YOLOv10n | 38.5 | 38.34 | 53.55 | 40.54 | 37.17 | 52.46 | 31.98 |
| YOLOv7-tiny | n/a | 37.13 | 54.64 | 39.41 | 36.65 | 50.50 | 31.09 |

## 2. How long does inference take?

The first table divides the median batch time by the number of images in the batch. In brackets: images per second and the speed-up over ONNX Runtime CUDA FP32 (ORT). The second table gives the time for the **whole request**. A batch of eight is not eight cameras running live.

### How to read the batch timings

A **batch** is the number of images in one inference request. It is not the number of cameras. These measurements use the **same TensorRT engine** at batch 1, 4 and 8; the engine's min/opt/max profile is 1/8/8. Batch 1 means one image was sent to that engine—not that a separate batch-1 engine was tested. The profile allows 1 through 8 images per request; 8 is the maximum and the build's optimization target, not a requirement to send eight.

Compare the two time columns carefully. **Whole-request time** is how long that batch took; **ms/image** divides that time by the number of images, showing average inference cost per image. For example, YOLOv10n FP16 takes **5.50 ms** for one image, or **30.10 ms** for eight. The average falls from 5.50 to **3.76 ms/image**, so batching improves throughput—but one image in the batch does not finish in 3.76 ms. A larger batch may also wait for more images to arrive; that waiting is not included in this benchmark.

For a **one-camera, one-frame request**, start with batch 1. The batch 8 result describes aggregate processing when eight images are available together; it does not mean one camera runs eight times faster. We did not compare this engine with an engine built to accept only one image per request, so these results cannot tell you whether that alternative would be faster.

The engine's dynamic batch profile is separate from **Triton's dynamic batching**, which schedules queued requests and may combine them. The model configuration enables that scheduler, but the raw latency test calls TensorRT directly, so it does not measure Triton's behavior or camera-to-result time. It uses synthetic GPU-resident inputs and excludes decoding, tracking and video output; for YOLOv7-tiny it also excludes the deployment engine’s EfficientNMS plugin. Latency uses rebuilt engines while the accuracy results are from an earlier run. Use the [live camera tests](../doc/benchmarks/capacity/README.md) for complete-pipeline performance and [measurement conditions](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md) for test details.

| Model | Batch 1 (ms per image) | Batch 4 (ms per image) | Batch 8 (ms per image) |
|---|---:|---:|---:|
| YOLO26n | 5.31 (188 img/s, 3.6x ORT) | 3.76 (266 img/s, 3.8x ORT) | 3.42 (292 img/s, 3.9x ORT) |
| YOLOv10n | 5.50 (182 img/s, 3.3x ORT) | 4.09 (244 img/s, 3.7x ORT) | 3.76 (266 img/s, 3.7x ORT) |
| YOLOv7-tiny | 8.40 (119 img/s, 2.9x ORT) | 5.31 (188 img/s, 4.3x ORT) | 4.74 (211 img/s, 4.1x ORT) |

**All runtimes and precisions** (median time of one request):

| Model | Batch | ONNX Runtime CUDA FP32 | TensorRT FP32 | TensorRT FP16 |
|---|---:|---:|---:|---:|
| YOLO26n | 1 | 19.0 ms | 12.2 ms | 5.3 ms |
| YOLO26n | 4 | 56.9 ms | 38.5 ms | 15.0 ms |
| YOLO26n | 8 | 106.1 ms | 72.0 ms | 27.4 ms |
| YOLOv10n | 1 | 18.4 ms | 12.7 ms | 5.5 ms |
| YOLOv10n | 4 | 61.4 ms | 40.3 ms | 16.4 ms |
| YOLOv10n | 8 | 112.2 ms | 76.5 ms | 30.1 ms |
| YOLOv7-tiny | 1 | 24.4 ms | 18.0 ms | 8.4 ms |
| YOLOv7-tiny | 4 | 90.8 ms | 64.5 ms | 21.3 ms |
| YOLOv7-tiny | 8 | 154.7 ms | 124.4 ms | 37.9 ms |

### See the trade-off

Lower amortized inference cost per image is better (left); higher mAP is better (right). The left chart does not show whole-batch response time or camera latency.

![Left: TensorRT FP16 milliseconds per image at batch 1, 4 and 8 for YOLO26n, YOLOv10n and YOLOv7-tiny. Right: measured COCO mAP 50:95 on 80 classes next to the published figure.](../doc/benchmarks/orin-nx-16gb-jp6.0/comparison.png)

### Does FP16 change detection quality?

Compare FP32 and FP16 scores at the **same batch size**. The difference is in mAP points, not percent speed-up.

| Model | Batch | mAP 50:95 TensorRT FP32 | mAP 50:95 TensorRT FP16 | Difference (points) |
|---|---:|---:|---:|---:|
| YOLO26n | 8 | 39.93 | 39.92 | -0.00 |
| YOLOv10n | 8 | 38.33 | 38.34 | +0.01 |
| YOLOv7-tiny | 8 | 37.15 | 37.13 | -0.02 |

## 3. How many live cameras does that mean?

Model throughput alone does not predict camera capacity. The pipeline must also decode the incoming video, track detections and, when requested, encode an output mosaic. The tests used one pipeline and **30 FPS H.264 RTSP cameras**. Each camera had to average at least **29 processed FPS** over 120 seconds after a 45-second warm-up. A count is reported only when it passed at least two attempts, failed none, and the next count failed at least once.

### EX5 and EX3 capacity results

The table uses 720p inputs. Output profiles enable the on-screen display (OSD). **Not qualified at 1** means the video failed a quality check at one camera even though detection kept up. **At least 1 (screened)** means one camera passed processing and video-quality checks; it is not the maximum camera count.

| Device | Model | No output | WebRTC | Smart Video | WebRTC + Smart Video |
|---|---|---:|---:|---:|---:|
| EX5, Orin NX 16 GB | YOLO26n | 7 | At least 1 (screened) | At least 1 (screened) | At least 1 (screened) |
| EX5, Orin NX 16 GB | YOLOv10n | 6† | At least 1 (screened) | At least 1 (screened) | At least 1 (screened) |
| EX5, Orin NX 16 GB | YOLOv7-tiny | 4 | At least 1 (screened) | At least 1 (screened) | At least 1 (screened) |
| EX3, Orin Nano 8 GB | YOLO26n | 5† | Not qualified at 1 | Not qualified at 1 | Not qualified at 1 |
| EX3, Orin Nano 8 GB | YOLOv10n | 5 | Not qualified at 1 | Not qualified at 1 | Not qualified at 1 |
| EX3, Orin Nano 8 GB | YOLOv7-tiny | 3 | Not qualified at 1 | Not qualified at 1 | Not qualified at 1 |

† EX5 YOLOv10n: seven 720p cameras passed 1 of 2 attempts. EX3 YOLO26n: six 720p cameras passed 3 of 7 attempts. The station reports explain how each count qualified and why a video-output check can fail while detection remains near 30 FPS.

In these 720p, OSD-on output checks, EX5 passed the one-camera WebRTC, Smart Video and combined checks; EX3 did not qualify those profiles. EX5 multi-camera output limits have not been established. These device results use different input scenes, so treat them as station-specific findings rather than a controlled hardware comparison. Earlier EX3 results used different checks and are listed separately in the [EX3 report](../doc/benchmarks/orin-nano-8gb-jp6.0/capacity.md#earlier-measurements-protocol-v1).

In the [matched one-camera comparison](../doc/benchmarks/capacity/matched.md), both stations processed about 30 FPS in all 33 cases. EX5 passed 28 of 30 cases with video output; its two misses were saved clips, not WebRTC delivery. EX3 passed two output cases: the 640×360 WebRTC mosaic with YOLO26n and YOLOv10n. At that size YOLOv7-tiny delivered 28.50 FPS, below the 29 FPS threshold. **Only the output size changed; the camera stayed at 720p30.** See the [OSD, bitrate and size settings](../doc/benchmarks/capacity/README.md#matched-one-camera-test) to reproduce the comparison.

## 4. Where do these results come from?

Accuracy and inference time were measured on EX5; live camera capacity and output quality were tested on both EX5 and EX3. The camera counts above used different scenes and are not a controlled, scene-identical comparison. For hardware and software versions, test conditions and trial evidence, see the [EX5 report](../doc/benchmarks/orin-nx-16gb-jp6.0/capacity.md), [EX3 report](../doc/benchmarks/orin-nano-8gb-jp6.0/capacity.md) and [EX5 measurement conditions](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md).

## 5. Repeat the model measurements (optional, station only)

Leave `MEASURE = False` to skip **all** downloads, installs and benchmarks below. To make new measurements, use App-Builder on an idle development station with internet access and room for COCO val2017 and the engines. Reserve the GPU rather than stopping someone else’s pipeline. Allow roughly 40–60 minutes per flavor plus the download.

The cells export models, build raw engines, time inference and score labelled images. Results go to `tmp/benchmarks`, not the reference directory. Use `LIMIT = 0` for a full COCO comparison; a subset is only a smoke test. These commands do not run the live camera-capacity tests.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
assert Path("tools/compare_models.py").is_file(), "Open this notebook from the repository (notebooks/ folder)"

MEASURE = False          # set to True to run the cells below
LIMIT = 0                # score only the first LIMIT images (0 = all 5,000); use e.g. 200 for a quick check
FLAVORS = ["yolov7", "yolov10", "yolo26"]

os.environ.update(MEASURE=str(int(MEASURE)), LIMIT=str(LIMIT), FLAVORS=" ".join(FLAVORS))

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
mkdir -p tmp/coco && cd tmp/coco
[ -d val2017 ] || { curl -fsSLO http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip && rm val2017.zip; }
[ -f annotations/instances_val2017.json ] || { curl -fsSLO http://images.cocodataset.org/annotations/annotations_trainval2017.zip \
    && unzip -q -o annotations_trainval2017.zip annotations/instances_val2017.json && rm annotations_trainval2017.zip; }
python3 -m pip install --user -r ../../tools/requirements-accuracy.txt
python3 -m pip install --user --no-deps -r ../../tools/requirements-export.txt
ls val2017 | wc -l

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
for flavor in $FLAVORS; do
    python3 tools/export_model.py "$flavor"
done

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
for flavor in $FLAVORS; do
    python3 tools/measure.py "$flavor" --annotations tmp/coco/annotations/instances_val2017.json \
        --images tmp/coco/val2017 --limit "$LIMIT" --results tmp/benchmarks
done

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
python3 tools/compare_models.py tmp/benchmarks

The next cell shows your report, `tmp/benchmarks/comparison.md`, and its chart. The tables above keep describing the reference runs. Live camera capacity is a separate test; see the [capacity test plan](../doc/benchmarks/capacity/README.md).

In [ ]:
from IPython.display import Image, Markdown, display

OWN = Path("tmp/benchmarks")
if (OWN / "comparison.md").is_file():
    display(Markdown((OWN / "comparison.md").read_text()))
    if (OWN / "comparison.png").is_file():
        display(Image(filename=str(OWN / "comparison.png")))
else:
    print("No report in tmp/benchmarks yet: set MEASURE = True and run the cells above.")